# S1_03 : Completion inference on frozen eligible samples

Run after S1_01 and S1_02. Residential and outpatient are separate treatment contrasts; the outcome is **recorded administrative completion** (`REASON=1`). This notebook never changes eligible episode membership. Frequency is outside this analysis because its observed outcome depends on post-admission matching.

Within each eligible completion sample, use four constrained chained-imputation chains (seeds 1722–1725), 20 iterations per chain, and save iterations 12, 14, 16, 18, and 20: 20 completed datasets per treatment contrast. Include treatment, outcome, and state-year in imputation, preserve observed values, and enforce the approved substance-count and employment/detail constraints. On **each** completed dataset independently, fit five-fold propensity and arm-specific outcome models, then estimate the overlap-weighted augmented contrast. Pool estimates across completed datasets; never average propensity scores first. The 2022 and 2023 episodes are pooled within each represented-state treatment contrast.

The reported state-score standard errors condition on the completed data and fitted models. Combining them with between-imputation variance gives a **descriptive conditional uncertainty summary**, not a validated confidence interval: only a small number of states are represented, episodes may repeat within patient, imputation assumes missing at random given included variables, and treatment may remain confounded. This notebook does not evaluate or refit a policy tree. The 2022 state crosswalks document coding differences; corresponding 2023 state mappings have not been verified. See [Leyrat et al. (2019)](https://pubmed.ncbi.nlm.nih.gov/28573919/) for within-imputation propensity analysis.

In [1]:
from pathlib import Path
from datetime import datetime
import gc, hashlib, json, os, platform, sys, time, warnings
import numpy as np
import pandas as pd
from scipy.stats import t as student_t
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/stage1_specifications.json').is_file())
HOME=ROOT/'S1';sys.path.insert(0,str(HOME/'helper'));sys.path.insert(0,str(ROOT))
from reporting import Review
from balance import balance_table,weight_summary
from src.causal_discovery.s1.development_inference import state_score_uncertainty
SPEC=json.loads((HOME/'helper/assets/sample_specification.json').read_text())
FIELDS=SPEC['fields'];PILOT=os.environ.get('S1_INFERENCE_PILOT')=='1'
CHAINS=1 if PILOT else 4
ITERATIONS=2 if PILOT else 20
SAVE_AT=(2,) if PILOT else (12,14,16,18,20)
SEEDS=[1722+i for i in range(CHAINS)]
FIXED_OUTPUTS=Path(os.environ.get('REPLICATION_S1_FIXED_OUTPUTS',ROOT/'S1/fixed_inputs')).resolve()
OUTPUTS=HOME/'outputs';REPORTS=HOME/'reports'
OUTPUTS.mkdir(parents=True,exist_ok=True);REPORTS.mkdir(parents=True,exist_ok=True)
def sha(path):
    with Path(path).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
COMPLETE_STATUSES={
 'S1_01_data_preparation':{'verified_samples_and_imputation_draws'},
 'S1_03_outpatient_data_preparation':{'candidate_membership_and_four_imputation_draws_verified','candidate_membership_and_four_imputation_draws_verified_overlap_review_required'},
}
def newest(prefix):
    for p in sorted(FIXED_OUTPUTS.glob(prefix+'_*'),reverse=True):
        manifest=p/'manifest.json'
        if manifest.is_file():
            m=json.loads(manifest.read_text())
            if m.get('status') in COMPLETE_STATUSES[prefix]:return p,m
    raise FileNotFoundError(f'No complete {prefix} run under fixed input folder: {FIXED_OUTPUTS}')
res,m_res=newest('S1_01_data_preparation')
out,m_out=newest('S1_03_outpatient_data_preparation')
assert m_res['status']=='verified_samples_and_imputation_draws'
assert m_out['status'].startswith('candidate_membership_and_four_imputation_draws_verified')
assert m_out['analysis_role']=='primary_14_state' and len(m_out['states'])==14
BRANCHES={
 'outpatient':{'source':out,'file':out/'completion_final_eligible.csv.gz',
               'n':next(x['n'] for x in m_out['samples'] if x['outcome']=='completion')},
 'residential':{'source':res,'file':res/'completion_final_eligible.csv.gz',
                'n':m_res['outputs']['completion_final_eligible.csv.gz']['n']}}
print('Fixed Stage 1 inputs:',FIXED_OUTPUTS)
print('New inference outputs:',OUTPUTS)
for branch,info in BRANCHES.items():
    assert info['file'].is_file()
    print(branch,info['source'].name,info['n'])
RUN=OUTPUTS/f"S1_03_completion_inference{'_pilot' if PILOT else ''}_{datetime.now().astimezone():%Y%m%d_%H%M%S_%f}"
RUN.mkdir()
review=Review(REPORTS,'S1_03_completion_inference'+('_pilot' if PILOT else ''))
review.note('Frozen completion membership; within-imputation overlap AIPW and conditional state-score uncertainty. No policy inference or validated causal interval.')
started=time.perf_counter()
def progress(**items):
    (RUN/'status.json').write_text(json.dumps({'status':'running','elapsed_seconds':time.perf_counter()-started,**items},indent=2)+'\n')
progress(stage='initialized')

Fixed Stage 1 inputs: /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/S1/outputs
New inference outputs: /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/outputs
outpatient S1_03_outpatient_data_preparation_20260922_130720_144842 386282
residential S1_01_data_preparation_20260922_113427_904266 241118


## Constrained categorical imputation

The same field set, joint employment/detail representation, structural feasibility rules, stochastic weighted categorical logistic fits, and ten-iteration minimum used in S1_01 are retained. Later iterations are thinned within four chains. Completed tables are analyzed immediately and not saved as additional large files; seeds, iterations, source hashes, diagnostics and draw estimates are recorded.

In [2]:
def load_branch(branch,info):
    d=pd.read_csv(info['file'])
    assert len(d)==info['n'] and d.episode_source_id.is_unique
    assert d.treatment.isin([0,1]).all() and d.outcome.isin([0,1]).all()
    assert d[['STFIPS','DISYR']].notna().all().all()
    if branch=='residential':
        assert sha(info['file'])==m_res['outputs'][info['file'].name]['sha256']
    else:
        expected=next(x for x in m_out['samples'] if x['outcome']=='completion')
        digest=hashlib.sha256(('\n'.join(sorted(d.episode_source_id.astype(str)))+'\n').encode()).hexdigest()
        assert digest==expected['membership_sha256']
    if PILOT:d=d.sample(n=min(4000,len(d)),random_state=991).sort_index().reset_index(drop=True)
    return d

def prepare_base(d):
    chain_fields=[f for f in FIELDS if f not in ('EMPLOY','DETNLF')]+['employment_joint','treatment','outcome']
    base=d[[f for f in chain_fields if f!='employment_joint']].astype(float)
    base['employment_joint']=(10*d.EMPLOY).astype(float)
    parent_four=d.EMPLOY.eq(4).fillna(False)
    base.loc[parent_four,'employment_joint']=(40+d.loc[parent_four,'DETNLF']).astype(float)
    base=base[chain_fields]
    slots=d[['SUB2','SUB3']]
    lower=slots.gt(1).fillna(False).sum(axis=1).to_numpy()
    upper=lower+slots.isna().sum(axis=1).to_numpy()
    assert (lower<=upper).all() and (upper<=2).all()
    targets=base.columns[base.isna().any()].tolist()
    return base,targets,lower,upper

def feasible(field,classes,rows,d,lower,upper):
    vals=np.asarray(classes)
    allowed=np.ones((len(rows),len(vals)),dtype=bool)
    if field=='additional_substances_total':
        allowed&=(vals[None,:]>=lower[rows,None])&(vals[None,:]<=upper[rows,None])
    if field=='employment_joint':
        parent=d.EMPLOY.to_numpy(dtype=float,na_value=np.nan)[rows]
        detail=d.DETNLF.to_numpy(dtype=float,na_value=np.nan)[rows]
        allowed&=np.isnan(parent[:,None])|(vals[None,:]//10==parent[:,None])
        known=np.isfinite(detail)&(detail>0)
        allowed[known]&=vals[None,:]==(40+detail[known,None])
        allowed[np.isfinite(detail)&(detail==0)]&=vals[None,:]<40
    return allowed

def categorical_draw(prob,classes,rng):
    assert np.isfinite(prob).all() and (prob>=0).all() and np.allclose(prob.sum(axis=1),1)
    u=rng.random(len(prob))
    choice=(u[:,None]>=np.cumsum(prob,axis=1)).sum(axis=1)
    assert (choice<len(classes)).all()
    return np.asarray(classes)[choice]

def imputed_draws(d,seed,branch,chain_id,trace_rows):
    rng=np.random.default_rng(seed)
    base,targets,lower,upper=prepare_base(d)
    z=base.copy()
    for field in targets:
        classes,counts=np.unique(base[field].dropna(),return_counts=True)
        rows=np.flatnonzero(base[field].isna())
        prob=feasible(field,classes,rows,d,lower,upper)*counts[None,:]
        assert (prob.sum(axis=1)>0).all(),field
        prob=prob/prob.sum(axis=1)[:,None]
        z.loc[base[field].isna(),field]=categorical_draw(prob,classes,rng)
    for iteration in range(1,ITERATIONS+1):
        progress(branch=branch,chain=chain_id,iteration=iteration,stage='imputation')
        for field in targets:
            observed=base[field].notna();rows=np.flatnonzero(~observed)
            x=z.drop(columns=field).astype(str)
            x['state_year']=x.STFIPS+'|'+x.DISYR
            x['treatment_outcome']=x.treatment+'|'+x.outcome
            classes=np.sort(base.loc[observed,field].unique())
            if len(classes)==1:prob=np.ones((len(rows),1))
            else:
                model=make_pipeline(OneHotEncoder(handle_unknown='ignore'),LogisticRegression(C=1,max_iter=2000))
                w=rng.exponential(size=observed.sum());w/=w.mean()
                with warnings.catch_warnings():
                    warnings.simplefilter('error',ConvergenceWarning)
                    model.fit(x.loc[observed],base.loc[observed,field].astype(int),logisticregression__sample_weight=w)
                classes=model[-1].classes_;prob=model.predict_proba(x.loc[~observed])
            prob*=feasible(field,classes,rows,d,lower,upper)
            assert (prob.sum(axis=1)>0).all(),field
            prob/=prob.sum(axis=1)[:,None]
            z.loc[~observed,field]=categorical_draw(prob,classes,rng)
        for field in targets:
            missing=base[field].isna()
            for category,share in z.loc[missing,field].value_counts(normalize=True).items():
                trace_rows.append({'branch':branch,'chain':chain_id,'iteration':iteration,
                                   'field':field,'category':category,'share':share})
        if iteration in SAVE_AT:
            assert not z.isna().any().any()
            for field in base:
                observed=base[field].notna()
                assert np.array_equal(z.loc[observed,field],base.loc[observed,field]),field
            assert z.employment_joint.isin([10,20,30,41,42,43,44,45]).all()
            assert z.additional_substances_total.between(lower,upper).all()
            complete=z.copy()
            complete['EMPLOY']=complete.employment_joint//10
            complete['DETNLF']=np.where(complete.EMPLOY.eq(4),complete.employment_joint%10,0)
            for field in ('EMPLOY','DETNLF'):
                observed=d[field].notna()
                assert np.array_equal(complete.loc[observed,field],d.loc[observed,field].astype(float))
            yield iteration,complete
            del complete;gc.collect()


## Estimation and conditional uncertainty

All propensity and outcome fits are recomputed within each completed dataset. Fixed five-fold assignments ensure that the analysis uses the same episodes and folds across draws. The overlap-weighted augmented score is normalized by mean `e(1-e)`. State-score standard errors condition on the supplied completed covariates and fitted models; their Rubin-style combination below is explicitly exploratory.
The propensity specification matches S1_01: state-year cells and their interactions with each admission covariate. The outcome models use the admission covariates and state/year main effects. The output includes the corresponding overlap-weighted unaugmented contrast to audit the augmentation.

In [3]:
def folds_for(d):
    groups=d.treatment.astype(int)
    assert groups.value_counts().min()>=5
    fold=np.full(len(d),-1,dtype=np.int8)
    for k,(_,test) in enumerate(StratifiedKFold(5,shuffle=True,random_state=720).split(d,groups)):
        fold[test]=k
    assert np.isin(fold,range(5)).all()
    return fold

def estimate_draw(complete,d,fold):
    covariates=complete[FIELDS].astype(str)
    cell=covariates.STFIPS+'|'+covariates.DISYR
    propensity_design=covariates.copy()
    propensity_design['state_year']=cell
    for field in FIELDS:
        if field not in ('STFIPS','DISYR'):
            propensity_design['state_year_by_'+field]=cell+'|'+covariates[field]
    X=OneHotEncoder(handle_unknown='ignore',dtype=np.float32).fit_transform(covariates)
    Xp=OneHotEncoder(handle_unknown='ignore',dtype=np.float32).fit_transform(propensity_design)
    t=d.treatment.to_numpy(int);y=d.outcome.to_numpy(int)
    e=np.empty(len(d));m0=np.empty(len(d));m1=np.empty(len(d))
    for k in range(5):
        train=fold!=k;test=fold==k
        assert set(t[train])=={0,1} and set(t[test])=={0,1}
        propensity=LogisticRegression(C=1,max_iter=10000,tol=1e-6).fit(Xp[train],t[train])
        outcome0=LogisticRegression(C=1,max_iter=2000).fit(X[train&(t==0)],y[train&(t==0)])
        outcome1=LogisticRegression(C=1,max_iter=2000).fit(X[train&(t==1)],y[train&(t==1)])
        e[test]=propensity.predict_proba(Xp[test])[:,1]
        m0[test]=outcome0.predict_proba(X[test])[:,1]
        m1[test]=outcome1.predict_proba(X[test])[:,1]
    assert np.isfinite(e).all() and ((e>0)&(e<1)).all()
    h=e*(1-e)
    dr=m1-m0+t*(y-m1)/e-(1-t)*(y-m0)/(1-e)
    score=h*dr/h.mean()
    ids=pd.Index(d.episode_source_id.to_numpy(),name='episode_source_id')
    unc=state_score_uncertainty(pd.Series(score,index=ids),pd.Series(d.STFIPS.to_numpy(),index=ids))
    w=np.where(t==1,1-e,e)
    ess=weight_summary(t,w).set_index('treatment').effective_sample_size
    dummy=pd.get_dummies(complete[FIELDS].astype(str),dtype=np.float32)
    balance=balance_table(dummy,t,w)
    state=pd.DataFrame({'state':d.STFIPS.to_numpy(),'score':score,'h':h}).groupby('state',sort=True).agg(
        n=('score','size'),score_sum=('score','sum'),h_sum=('h','sum'))
    state['h_share']=state.h_sum/state.h_sum.sum()
    result={'estimate':float(score.mean()),
            'ipw_contrast':float(np.average(y[t==1],weights=w[t==1])-np.average(y[t==0],weights=w[t==0])),
            'outcome_model_component':float(np.mean(h*(m1-m0))/h.mean()),
            'conditional_state_se':unc['state_score_se'],
            'states':unc['clusters'],'n':len(d),'max_state_h_share':float(state.h_share.max()),
            'test_propensity_tail_pct':float(100*((e<.05)|(e>.95)).mean()),
            'max_abs_weighted_smd':float(balance.abs_smd.max()),
            'control_ess':float(ess.loc[0]),'treated_ess':float(ess.loc[1])}
    del X,Xp,dummy,balance;gc.collect()
    return result,state.reset_index()

def scalar_rhat(draws):
    wide=draws.pivot(index='iteration',columns='chain',values='estimate')
    assert wide.shape==(len(SAVE_AT),CHAINS) and wide.notna().all().all()
    if CHAINS<2 or len(SAVE_AT)<2:return np.nan
    within=wide.var(axis=0,ddof=1).mean()
    between=len(wide)*wide.mean(axis=0).var(ddof=1)
    if within==0:return np.inf if between>0 else 1.0
    return float(np.sqrt(((len(wide)-1)/len(wide)*within+between/len(wide))/within))

def pool(draws):
    m=len(draws);assert m==CHAINS*len(SAVE_AT)
    q=draws.estimate.to_numpy(float);u=np.square(draws.conditional_state_se.to_numpy(float))
    within=float(u.mean());between=float(q.var(ddof=1)) if m>1 else np.nan
    total=within+(1+1/m)*between if m>1 else np.nan
    rhat=scalar_rhat(draws) if not PILOT else np.nan
    df_mi=(m-1)*(1+within/((1+1/m)*between))**2 if between>0 else np.inf
    df_ref=min(float(draws.states.min()-1),df_mi)
    half=float(student_t.ppf(.975,df_ref)*np.sqrt(total)) if np.isfinite(total) else np.nan
    return {'draws':m,'estimate':float(q.mean()),'within_state_score_variance':within,
            'between_imputation_variance':between,'descriptive_total_se':float(np.sqrt(total)) if np.isfinite(total) else np.nan,
            'conditional_interval_lower':float(q.mean()-half),'conditional_interval_upper':float(q.mean()+half),
            'reference_df':df_ref,'estimate_rhat':rhat,
            'max_score_tail_pct':float(draws.test_propensity_tail_pct.max()),
            'max_weighted_smd':float(draws.max_abs_weighted_smd.max()),
            'status':'pilot_only' if PILOT else 'exploratory_conditional_uncertainty_not_validated_causal_interval'}

# Small deterministic score check: randomized balanced data with zero residuals has known contrast.
y=np.array([0,1,0,1]);t=np.array([0,1,0,1]);e=np.full(4,.5);m0=np.zeros(4);m1=np.ones(4)
assert np.isclose(np.mean(e*(1-e)*(m1-m0+t*(y-m1)/e-(1-t)*(y-m0)/(1-e)))/np.mean(e*(1-e)),1)
print('Overlap AIPW score self-check passed')

Overlap AIPW score self-check passed


## Execute: outpatient first, then residential

The pilot environment flag (`S1_INFERENCE_PILOT=1`) runs a 4,000-episode engineering check and cannot produce a final result. A normal run uses the full, hash-verified fixed membership. Partial draw tables and `status.json` are updated during long execution.

In [4]:
draw_rows=[];state_rows=[];trace_rows=[];pooled_rows=[]
for branch,info in BRANCHES.items():
    d=load_branch(branch,info);fold=folds_for(d)
    review.note(f'## {branch}: {len(d):,} completion episodes')
    for chain_id,seed in enumerate(SEEDS,1):
        for iteration,complete in imputed_draws(d,seed,branch,chain_id,trace_rows):
            progress(branch=branch,chain=chain_id,iteration=iteration,stage='estimating_draw',completed_draws=len(draw_rows))
            result,state=estimate_draw(complete,d,fold)
            result.update(branch=branch,chain=chain_id,seed=seed,iteration=iteration,
                          source_file_sha256=sha(info['file']))
            draw_rows.append(result)
            state.insert(0,'iteration',iteration);state.insert(0,'chain',chain_id);state.insert(0,'branch',branch)
            state_rows.append(state)
            pd.DataFrame(draw_rows).to_csv(RUN/'draw_estimates.csv',index=False)
            pd.concat(state_rows,ignore_index=True).to_csv(RUN/'state_draw_scores.csv',index=False)
            pd.DataFrame(trace_rows).to_csv(RUN/'imputation_traces.csv',index=False)
            progress(branch=branch,chain=chain_id,iteration=iteration,stage='draw_saved',completed_draws=len(draw_rows))
            print(branch,'chain',chain_id,'iteration',iteration,'estimate',round(result['estimate'],5),flush=True)
            del complete,state;gc.collect()
    branch_draws=pd.DataFrame(draw_rows).query('branch == @branch').copy()
    assert len(branch_draws)==CHAINS*len(SAVE_AT)
    summary=pool(branch_draws);summary['branch']=branch;pooled_rows.append(summary)
    review.table(f'{branch} draw estimates',branch_draws)
    review.table(f'{branch} pooled conditional summary',pd.DataFrame([summary]))
    review.mark(f'Complete {branch} imputations and estimates')
    del d,fold;gc.collect()


outpatient chain 1 iteration 12 estimate -0.01859
outpatient chain 1 iteration 14 estimate -0.01849
outpatient chain 1 iteration 16 estimate -0.01856
outpatient chain 1 iteration 18 estimate -0.01868
outpatient chain 1 iteration 20 estimate -0.0185
outpatient chain 2 iteration 12 estimate -0.01872
outpatient chain 2 iteration 14 estimate -0.01862
outpatient chain 2 iteration 16 estimate -0.01861
outpatient chain 2 iteration 18 estimate -0.01856
outpatient chain 2 iteration 20 estimate -0.0186
outpatient chain 3 iteration 12 estimate -0.01837
outpatient chain 3 iteration 14 estimate -0.01848
outpatient chain 3 iteration 16 estimate -0.01836
outpatient chain 3 iteration 18 estimate -0.01837
outpatient chain 3 iteration 20 estimate -0.01836
outpatient chain 4 iteration 12 estimate -0.01846
outpatient chain 4 iteration 14 estimate -0.01841
outpatient chain 4 iteration 16 estimate -0.01856
outpatient chain 4 iteration 18 estimate -0.01854
outpatient chain 4 iteration 20 estimate -0.01869
re

In [5]:
draws=pd.DataFrame(draw_rows);pooled=pd.DataFrame(pooled_rows);traces=pd.DataFrame(trace_rows)
draws.to_csv(RUN/'draw_estimates.csv',index=False)
pooled.to_csv(RUN/'pooled_summary.csv',index=False)
traces.to_csv(RUN/'imputation_traces.csv',index=False)
stability=traces.loc[traces.iteration.ge(max(1,ITERATIONS-4))].groupby(['branch','chain','field','category']).share.agg(lambda x:x.max()-x.min()).reset_index(name='last_five_iteration_range')
stability.to_csv(RUN/'trace_stability.csv',index=False)
review.table('Trace stability, largest category-share changes',stability.sort_values('last_five_iteration_range',ascending=False).head(30))
review.table('Pooled descriptive conditional results',pooled)
review.note('Intervals condition on completed covariates and fitted nuisance models, and are not validated causal confidence intervals. MAR, comparable administrative outcome coding, state dependence, and unmeasured confounding remain substantive assumptions.')
review.mark('Save summaries and audit')
report=review.export()
manifest={'status':'pilot_only' if PILOT else 'exploratory_conditional_inference_complete_not_validated_for_causal_claims',
 'created_at':datetime.now().astimezone().isoformat(),
 'sources':{branch:{'run':info['source'].name,'source_manifest_sha256':sha(info['source']/'manifest.json'),
                    'file':str(info['file'].relative_to(ROOT)),'file_sha256':sha(info['file']),
                    'n_full':info['n']} for branch,info in BRANCHES.items()},
 'design':{'outcome':'recorded administrative completion REASON=1','treatment_contrasts':'separate residential and outpatient',
           'chains':CHAINS,'seeds':SEEDS,'iterations':ITERATIONS,'saved_iterations':SAVE_AT,
           'nuisance_folds':5,'within_imputation_analysis':True,'target':'e(X)*(1-e(X)) overlap population',
           'pilot':PILOT,'inference':'descriptive conditional state-score SE plus between-imputation variance'},
 'software':{'python':platform.python_version(),'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__},
 'results':pooled.to_dict(orient='records'),
 'files':{p.name:sha(p) for p in RUN.iterdir() if p.is_file() and p.name!='status.json'},
 'summary_report':str(report.relative_to(ROOT)),
 'limits':['Only represented states and recorded administrative completion are studied.',
           'State-score variance conditions on imputation and fitted nuisance models; Rubin-style pooling is exploratory.',
           'Multiple episodes from one person cannot be linked in public data.',
           'Missing-at-random and no-unmeasured-confounding assumptions cannot be verified.',
           '2023 state-specific completion crosswalks have not been verified.',
           'Neither a causal effect nor a clinical policy rule is established.']}
(RUN/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
(RUN/'status.json').write_text(json.dumps({'status':manifest['status'],'elapsed_seconds':time.perf_counter()-started,
                                          'completed_draws':len(draws)},indent=2)+'\n')
print('Saved:',RUN/'manifest.json')
print('Review:',report)
display(pooled)

Saved: /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/outputs/S1_03_completion_inference_20260926_202737_859797/manifest.json
Review: /Users/researcher/code_projects/Causal Discovery /code/project_code/notebooks/code_0927/S1/reports/S1_03_completion_inference_20260927_034406_996590.html


,draws,estimate,within_state_score_variance,between_imputation_variance,descriptive_total_se,conditional_interval_lower,conditional_interval_upper,reference_df,estimate_rhat,max_score_tail_pct,max_weighted_smd,status,branch
0,20,-0.018526,0.000531,1.289438e-08,0.023038,-0.068296,0.031244,13.0,1.583559,7.664349,0.001041,exploratory_conditional_uncertainty_not_valida...,outpatient
1,20,-0.150956,0.000711,1.761831e-08,0.026656,-0.207194,-0.094717,17.0,0.915987,21.694357,0.009966,exploratory_conditional_uncertainty_not_valida...,residential
